# **Raw Data QA**

# All Imports Here

In [ ]:
import os
glb_pth = 'd:\\GitHub\\SaaS_Product_Analysis'
os.chdir(glb_pth)

import re
import pandas as pd
import json
from pathlib import Path

# Importing Datasets

In [2]:
def load_product_reviews(product_path, file_name, product_name=None):
    """
    Read all raw review JSON files for one product,
    extract review-level data plus file-level metadata,
    and combine everything into one DataFrame.
    """

    chunk_dfs = []

    for json_file in product_path.rglob(file_name):

        with open(json_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        # review-level data
        reviews = raw_data.get("reviews", [])

        df = pd.json_normalize(reviews)

        # file-level metadata
        df["appId"] = raw_data.get("appId")
        df["reviewCount"] = raw_data.get("reviewCount")
        df["pagesRequested"] = raw_data.get("pagesRequested")
        df["scrapedAt"] = raw_data.get("scrapedAt")

        if product_name is not None:
            df["product"] = product_name

        chunk_dfs.append(df)

    if not chunk_dfs:
        return pd.DataFrame()

    return pd.concat(chunk_dfs, ignore_index=True)

In [3]:
base_path = Path(glb_pth) / "data"

zoom_df = load_product_reviews(
    base_path / "zoom",
    "zoom_raw_reviews.json",
    product_name="Zoom"
)

meet_df = load_product_reviews(
    base_path / "google_meet",
    "meet_raw_reviews.json",
    product_name="Google Meet"
)

webex_df = load_product_reviews(
    base_path / "cisco_webex",
    "webex_raw_reviews.json",
    product_name="Cisco Webex"
)

teams_df = load_product_reviews(
    base_path / "microsoft_teams",
    "teams_raw_reviews.json",
    product_name="Microsoft Teams"
)

# Inspecting Duplicate Reviews

In [4]:
zoom_df.duplicated().sum()

np.int64(0)

# Inspecting Null Values

In [5]:
webex_df.isnull().sum()


reviewId            0
rating              0
reviewer            0
date                0
reviewedIn          0
body                0
userImage           0
position            0
helpfulCounts       0
appId               0
appVersion        861
timestamp           0
language            0
reviewCount         0
pagesRequested      0
scrapedAt           0
product             0
dtype: int64

# Inspecting Review Volume

In [6]:
print(f"Review volume of Zoom: {len(zoom_df)}")
print(f"Review volume of Google Meet: {len(meet_df)}")
print(f"Review volume of Microsoft Teams: {len(teams_df)}")
print(f"Review volume of Cisco Webex: {len(webex_df)}")

Review volume of Zoom: 6000
Review volume of Google Meet: 6000
Review volume of Microsoft Teams: 6000
Review volume of Cisco Webex: 6000


# Inspecting Date Ranges

In [7]:
print(f"Date Range of Zoom: \n{min(zoom_df['date'])}   to   {max(zoom_df['date'])}")
print(f"\nDate Range of Google Meet: \n{min(meet_df['date'])}   to   {max(meet_df['date'])}")
print(f"\nDate Range of Microsoft Teams: \n{min(teams_df['date'])}   to   {max(teams_df['date'])}")
print(f"\nDate Range of Cisco Webex: \n{min(webex_df['date'])}   to   {max(webex_df['date'])}")

Date Range of Zoom: 
2026-02-06   to   2026-08-26

Date Range of Google Meet: 
2026-03-24   to   2026-08-26

Date Range of Microsoft Teams: 
2026-04-13   to   2026-08-26

Date Range of Cisco Webex: 
2024-07-20   to   2026-08-25


# Checking If Ratings are Valid

In [8]:
webex_df['rating'].unique()

array([4, 5, 1, 2, 3])

# Checking Date Validity

In [9]:
valid_dates = zoom_df['date']

valid_dates = pd.to_datetime(valid_dates)

valid_dates.dtype

dtype('<M8[us]')

# Separating Table

In [10]:
prod_review_tbl_cols = [
    'reviewId',
    'rating',
    'date',
    'body',
    'appVersion',
    'timestamp'
]

scrap_info_tbl_cols = [
    'product',
    'appId',
    'pagesRequested',
    'reviewCount',
    'scrapedAt'
]

In [11]:
# prod review tables/dataframes
zoom_reviews_df = zoom_df[prod_review_tbl_cols]
meet_reviews_df = meet_df[prod_review_tbl_cols]
teams_reviews_df = teams_df[prod_review_tbl_cols]
webex_reviews_df = webex_df[prod_review_tbl_cols]

In [12]:
webex_reviews_df.head()

,reviewId,rating,date,body,appVersion,timestamp
0,5357375b-b0f3-4911-8d9f-e578967960c1,4,2026-08-25,less used than zooms,45.3.0,1787681584
1,df121414-03e3-4dcf-91d3-9dcb7c265822,5,2026-08-25,Better,45.3.0,1787654961
2,a012b8ec-6102-4467-a9bf-a659ca645397,1,2026-08-25,it's a very bad app. More space needed and aft...,NaN,1787634361
3,62dab9ed-d87c-4771-a7a2-23339816a3f8,5,2026-08-24,very good app for us,NaN,1787544185
4,49290776-d039-45f6-a923-121ce46c2a12,1,2026-08-23,Not useful for the host. You can't schedule or...,45.3.0,1787447987


In [13]:
# scraping info tables/dataframes
scrap_info_df = pd.concat([
    zoom_df[scrap_info_tbl_cols].drop_duplicates(),
    meet_df[scrap_info_tbl_cols].drop_duplicates(),
    teams_df[scrap_info_tbl_cols].drop_duplicates(),
    webex_df[scrap_info_tbl_cols].drop_duplicates(),
], ignore_index=True)

scrap_info_df = scrap_info_df.sort_values("scrapedAt").reset_index(drop=True)

scrap_info_df.insert(0, "scrapId", scrap_info_df.index)

scrap_info_df

,scrapId,product,appId,pagesRequested,reviewCount,scrapedAt
0,0,Zoom,us.zoom.videomeetings,10,2000,2026-08-27T07:03:48.665Z
1,1,Zoom,us.zoom.videomeetings,5,1000,2026-08-27T07:09:14.503Z
2,2,Zoom,us.zoom.videomeetings,5,1000,2026-08-27T07:12:23.153Z
3,3,Zoom,us.zoom.videomeetings,10,2000,2026-08-27T07:32:50.746Z
4,4,Microsoft Teams,com.microsoft.teams,10,2000,2026-08-27T07:35:08.702Z
5,5,Microsoft Teams,com.microsoft.teams,10,2000,2026-08-27T07:35:51.938Z
6,6,Microsoft Teams,com.microsoft.teams,5,1000,2026-08-27T07:36:44.693Z
7,7,Microsoft Teams,com.microsoft.teams,5,1000,2026-08-27T07:37:56.180Z
8,8,Google Meet,com.google.android.apps.tachyon,10,2000,2026-08-27T07:41:04.303Z
9,9,Google Meet,com.google.android.apps.tachyon,10,2000,2026-08-27T07:41:44.160Z


# Treating Timestamp

In [14]:
zoom_reviews_df['timestamp'] = pd.to_datetime(
    zoom_reviews_df['timestamp'],
    unit='s'
)

teams_reviews_df['timestamp'] = pd.to_datetime(
    teams_reviews_df['timestamp'],
    unit='s'
)

meet_reviews_df['timestamp'] = pd.to_datetime(
    meet_reviews_df['timestamp'],
    unit='s'
)

webex_reviews_df['timestamp'] = pd.to_datetime(
    webex_reviews_df['timestamp'],
    unit='s'
)


In [15]:
zoom_reviews_df.drop(columns='date', inplace=True)
teams_reviews_df.drop(columns='date', inplace=True)
meet_reviews_df.drop(columns='date', inplace=True)
webex_reviews_df.drop(columns='date', inplace=True)

In [16]:
zoom_reviews_df.head()

,reviewId,rating,body,appVersion,timestamp
0,0dcdf6d4-dfa0-4401-b3d4-d9dbc27c825f,1,👎👎👎👎👎,NaN,2026-08-26 05:50:54
1,d70e1f1c-83fc-495b-8035-3eb460431f82,1,Can't login after resignation,NaN,2026-08-26 05:50:12
2,ad750e18-d0d4-40cc-8797-26e77acdbfe4,1,Worst app,NaN,2026-08-26 05:49:41
3,359d855e-2a8b-4313-8120-91dff1c872ef,1,Can't login,NaN,2026-08-26 05:49:09
4,dae23cb0-c77e-4f78-89ea-72b01e4c7799,1,Can't login after resignation,NaN,2026-08-26 05:48:36


# Preprocessing Review Texts ('body' column)

In [17]:
zoom_reviews_df.head()

,reviewId,rating,body,appVersion,timestamp
0,0dcdf6d4-dfa0-4401-b3d4-d9dbc27c825f,1,👎👎👎👎👎,NaN,2026-08-26 05:50:54
1,d70e1f1c-83fc-495b-8035-3eb460431f82,1,Can't login after resignation,NaN,2026-08-26 05:50:12
2,ad750e18-d0d4-40cc-8797-26e77acdbfe4,1,Worst app,NaN,2026-08-26 05:49:41
3,359d855e-2a8b-4313-8120-91dff1c872ef,1,Can't login,NaN,2026-08-26 05:49:09
4,dae23cb0-c77e-4f78-89ea-72b01e4c7799,1,Can't login after resignation,NaN,2026-08-26 05:48:36


In [18]:
def preprocess_for_bert(text_series: pd.Series) -> pd.Series:
    """
    Light preprocessing for BERT-based sentiment analysis.
    """

    def clean_text(text):
        if pd.isna(text):
            return pd.NA

        text = str(text)

        # remove URLs
        text = re.sub(r"https?://\S+|www\.\S+", "", text)

        # remove HTML tags
        text = re.sub(r"<[^>]+>", "", text)

        # normalize whitespace
        text = re.sub(r"[\n\r\t]+", " ", text)
        text = re.sub(r"\s+", " ", text)

        text = text.strip()

        return text if text else pd.NA

    return text_series.apply(clean_text)

In [19]:
zoom_reviews_df['body'] = preprocess_for_bert(zoom_reviews_df['body'])
zoom_reviews_df.head()

,reviewId,rating,body,appVersion,timestamp
0,0dcdf6d4-dfa0-4401-b3d4-d9dbc27c825f,1,👎👎👎👎👎,NaN,2026-08-26 05:50:54
1,d70e1f1c-83fc-495b-8035-3eb460431f82,1,Can't login after resignation,NaN,2026-08-26 05:50:12
2,ad750e18-d0d4-40cc-8797-26e77acdbfe4,1,Worst app,NaN,2026-08-26 05:49:41
3,359d855e-2a8b-4313-8120-91dff1c872ef,1,Can't login,NaN,2026-08-26 05:49:09
4,dae23cb0-c77e-4f78-89ea-72b01e4c7799,1,Can't login after resignation,NaN,2026-08-26 05:48:36


In [20]:
webex_reviews_df.isnull().sum()

reviewId        0
rating          0
body            0
appVersion    861
timestamp       0
dtype: int64

In [21]:
teams_reviews_df.drop_duplicates(inplace=True)
zoom_reviews_df.drop_duplicates(inplace=True)
meet_reviews_df.drop_duplicates(inplace=True)
webex_reviews_df.drop_duplicates(inplace=True)

In [22]:
meet_reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype        
---  ------      --------------  -----        
 0   reviewId    6000 non-null   str          
 1   rating      6000 non-null   int64        
 2   body        6000 non-null   str          
 3   appVersion  5580 non-null   str          
 4   timestamp   6000 non-null   datetime64[s]
dtypes: datetime64[s](1), int64(1), str(3)
memory usage: 234.5 KB


In [23]:
scrap_info_df['scrapedAt'] = pd.to_datetime(scrap_info_df['scrapedAt'])
scrap_info_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   scrapId         16 non-null     int64              
 1   product         16 non-null     str                
 2   appId           16 non-null     str                
 3   pagesRequested  16 non-null     int64              
 4   reviewCount     16 non-null     int64              
 5   scrapedAt       16 non-null     datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), int64(3), str(2)
memory usage: 900.0 bytes


# Saving Reviews and Scrap Info Tables

In [24]:
zoom_reviews_df.to_csv(f'{glb_pth}\\data\\cleaned\\zoom_reviews_clean.csv')
meet_reviews_df.to_csv(f'{glb_pth}\\data\\cleaned\\meet_reviews_clean.csv')
teams_reviews_df.to_csv(f'{glb_pth}\\data\\cleaned\\teams_reviews_clean.csv')
webex_reviews_df.to_csv(f'{glb_pth}\\data\\cleaned\\webex_reviews_clean.csv')

scrap_info_df.to_csv(f'{glb_pth}\\data\\cleaned\\scrap_info_clean.csv')

# Calculating QA Metrics

- Rows before cleaning
- Rows after cleaning
- Rows removed
- Duplicate reduction
- Completeness
- Uniqueness
- Validity
- Row retention

In [25]:
def cleaning_validation(unclean_df, clean_df):
    unclean_len = len(unclean_df)
    clean_len = len(clean_df)

    # rows before cleaning
    row_bef_clean = unclean_len

    # rows after cleaning
    row_aft_clean = clean_len

    # rows removed
    row_rem = row_bef_clean - row_aft_clean

    # duplicate reduction
    dup_unclean = unclean_df.duplicated().sum()
    dup_clean = clean_df.duplicated().sum()
    duplicate_reduc = dup_unclean - dup_clean

    # completeness
    total_cells = clean_df.size
    non_na_vals = clean_df.notna().sum().sum()

    completeness_score = (
        non_na_vals / total_cells
    ) * 100.0

    # uniqueness
    uniqueness_score = (
        clean_df['reviewId'].nunique() / clean_df['reviewId'].notna().sum()
    ).mean() * 100.0

    # validity 
    validity = 100.0

    # row retention percentage
    row_retention = (
        clean_len / unclean_len
    ) * 100.0

    return (
        row_bef_clean,
        row_aft_clean,
        row_rem,
        duplicate_reduc,
        completeness_score,
        uniqueness_score,
        validity,
        row_retention
    )

In [26]:
cols = ['rows_before_clean', 'rows_after_clean', 'rows_rem', 'duplicate_reduc',
    'completeness', 'uniqueness', 'validity', 'row_retention']

zoom_validation_df = pd.DataFrame(
    [dict(zip(cols, cleaning_validation(zoom_df, zoom_reviews_df)))]
)

teams_validation_df = pd.DataFrame(
    [dict(zip(cols, cleaning_validation(teams_df, teams_reviews_df)))]
)

meet_validation_df = pd.DataFrame(
    [dict(zip(cols, cleaning_validation(meet_df, meet_reviews_df)))]
)

webex_validation_df = pd.DataFrame(
    [dict(zip(cols, cleaning_validation(webex_df, webex_reviews_df)))]
)

In [27]:
teams_validation_df

,rows_before_clean,rows_after_clean,rows_rem,duplicate_reduc,completeness,uniqueness,validity,row_retention
0,6000,5999,1,0,96.739457,100.0,100.0,99.983333


# Exporting Data Validation Tables / Dataframes

In [28]:
zoom_validation_df.to_csv(f'{glb_pth}\\data\\cleaned\\zoom_data_validation.csv')
teams_validation_df.to_csv(f'{glb_pth}\\data\\cleaned\\teams_data_validation.csv')
meet_validation_df.to_csv(f'{glb_pth}\\data\\cleaned\\meet_data_validation.csv')
webex_validation_df.to_csv(f'{glb_pth}\\data\\cleaned\\webex_data_validation.csv')